# ROI | Draw and Reuse Regions of Interest
-----------------------------------------

Use movement's napari interface to draw named static regions on the matching video frame, then reload the saved GeoJSON with movement.roi. This is separate from the geometry calculations.

## Setup and Configuration

In [ ]:
##### Imports ##########################################################################

#==== Generic Imports ==================================================================
from pathlib import Path
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

#==== Movement Imports =================================================================
import movement.kinematics
import movement.utils.vector

#==== Data-Conduit Imports =============================================================
from data_conduit.datastructures import slice_stream, slice_stream_per_trial
from data_conduit.integrations.DLC.pose import pose_to_movement
from data_conduit.refactor_qc import (
    TRAINING_FIRST_MID_LAST_DETAIL,
    training_spec,
    filter_trials,
)

#==== Figure-Project Imports ===========================================================
from movement_figures.data_template.loading import build_datastructure
from movement_figures.misc import visual_streams_check, configure_simplified_trial_df
from movement_figures.timeseries_template.annotations import (
    AnnotationResult,
    annotate_qc_timeseries,
)

pd.set_option("display.max_columns", None)                    # Show the trial boundary columns.
import movement.roi

from movement_figures.video import read_session_video_frame

In [ ]:
##### Setup Parameters #################################################################

# === 1| Select Mice, Days and Recording Folders Before Reading Files =============
ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training")

LEVEL_SELECTORS = {
    'l0_selector': 'MR_M01569515', 
    'l1_selector': 'Day21'
    }                                                                                       # Example choices belong here: l0_selector and l1_selector.
# INCLUDE_SESSIONS = ('2026-06-21T093349Z',)                                                # Optional list of recording folder names; None keeps selected folders.
# STREAMS = ('trials', 'events', 'video', 'dlc')                                            # These figures require events/trials and aligned pose; other sources remain available.


#=== 2| Set the trial filtering parameters for the experiment protocol ============
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)                                        # Flattens `{training_detail}` table into per-session spec. 

# included_sessions = training_session_names(spec)                                          # Returns a list of session folder names that match the spec.
#   Not used here, as sessions have been filtered in another way. If other sessions 
#   not in the above spec are used, then applying the `filter_trials` function with 
#   the above spec could result in incorrect filtering.

#==== Choose the Recording and Samples =================================================
SESSION_ID = None                                            # Set a folder ID when several recordings are loaded.
INDIVIDUAL = "individual_0"
TRIAL_ROWS = slice(0, 8)                                      # Row positions after protocol filtering; None uses all samples.
ROIS_FILE = Path("arena.geojson")                         # Save this same file from the movement GUI.

In [ ]:
##### Load DataStructure ###############################################################

#=== Load data ========================================================================

fig_ds_obj = build_datastructure(
    root               = ROOT,                                                                 # Directory above the named hierarchy levels.
    level_names        = ("mouseID", "day"),                                                   # Lab layout: root / mouseID / day / session.
    streams            = ("trials", "session_settings", "video", "dlc"),                       # Data streams to be included in the datastructure.
    include            = None,                                                                 # Keep these session folder names; None keeps all.
    exclude            = None,                                                                 # Alternatively omit these session folder names.
    raw_events         = True,                                                                 # Convenience alias for adding "events" to the requested sources.
    device_yaml        = "./device.yml",                                                       # Lab Nosepoke board register definitions.
    soundcard_yaml     = "./soundcard.yml",                                                    # Lab SoundCard register definitions.
    nosepoke_count     = 18,                                                                   # Port count used by the existing Q_C parser.
    trial_start_buffer = 0.0,                                                                  # Preserve the existing lab parser setting.
    dlc_file_format     = "auto",                                                              # Existing reader prefers CSV, otherwise HDF5.
    trial_spec = None,                                                                         # Optional experiment rules; None keeps the existing Q_C specification. 
    #    Not used here, as sessions are already selected. 
    #    If session is not one in the spec, then subsequent use of 
    #   `filter_trials` to apply correct experiment protocol rules could be incorrect.

    **LEVEL_SELECTORS,
)


#=== Select and load the streams of interest from datastructure. ======================
fig_ds_obj.select()
streams = fig_ds_obj.load()                                                                    # Load is required to access the selected streams.

#=== Create convenient compact trial dataframe without some unnecessary columns. ======
streams['compact_trials'] = configure_simplified_trial_df(              
    trial_df = streams['trials'],
    modify_in_place = False,                                                                   # Whether to modify the trial dataframe in place or return a new dataframe. 
)
'''More useful for visual inspection and analysis. 
Check the misc.py file for the default columns to keep and drop.'''


#=== Apply Trial Filtering Protocol to construct new stream objects ===================

# Full filtered trials
streams['filtered_trials'] = filter_trials(
    trials = streams['trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

# Filtered compact trials 
streams['filtered_compact_trials'] = filter_trials(
    trials = streams['compact_trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

#=== Merge Pose & Confidence Streams ==================================================
streams['dlc:movement'] = pose_to_movement(
    slice_stream(streams["dlc:position"],   
                #selectors={"session": session_id}
    ),
    slice_stream(streams["dlc:confidence"], 
                 #selectors={"session": session_id}
    ),
)

#=== Visual streams check =============================================================
visual_streams_check(
    loaded_ds_object = streams,                                                                # Loaded dataset object containing all streams of interest.
    show_stream_shapes = True,                                                                 # Whether to display the shapes of the streams.
    show_stream_dtypes = True,                                                                 # Whether to display the data types of the streams.
    display_streams = False                                                                    # Whether to display the actual stream data as a table.
)

## Select the Recording and Trials

In [ ]:
##### Select the Recording and Optional Trial Rows ######################################

#==== Keep One Recording Before Calculating Any Time Derivatives ========================
sessionID = SESSION_ID
if sessionID is None:
    sessionID = np.unique(streams["dlc:movement"]["session"].values).item()

recording = slice_stream(
    stream=streams["dlc:movement"],
    selectors={"session": sessionID},                        # Do not mix clocks from different recordings.
)
position = recording["position"].sel(individual=INDIVIDUAL, drop=True)
position = position.sel(space=["x", "y"])

#==== Keep the Matching Protocol-Filtered Trial Table ===================================
recording_trials = slice_stream(
    stream=streams["filtered_trials"],
    selectors={"session": sessionID},
)
selected_trials = None
if TRIAL_ROWS is not None:
    selected_trials = recording_trials.iloc[TRIAL_ROWS].copy()

In [ ]:
##### Read the Matching Video Background ################################################

background_frame, video_path = read_session_video_frame(
    fig_ds_obj.sessions[sessionID].path,                     # Same recording and undistorted pixel coordinates as position.
)

## Open the Native Drawing Interface

This optional notebook requires the movement napari plugin and a working local Qt display. It is not needed to run any of the ordinary plotting helpers. Run the next cell locally, draw the measured arena outline, and name it `arena`. A separate named `target` region can be used for nearest-region egocentric angles.

In the movement dock, expand **Define regions of interest**, create a region layer, draw the shapes, name them, and save the region layer as `arena.geojson` in this notebook's folder. The spatial notebooks refer to `../ROI/arena.geojson`; change those paths when running from a different working directory. Keep the x/y coordinates of the unscaled video.

In [ ]:
##### Open the Video Frame in movement's ROI Interface ##################################

%gui qt
import napari

viewer = napari.Viewer()
viewer.add_image(background_frame, name=f"Video | {sessionID}")
viewer.window.add_plugin_dock_widget("movement", "movement")

## Reload the Saved Geometry

Run this after saving the named regions in the GUI. A drawn line/path becomes `LineOfInterest`; polygons and rectangles become `PolygonOfInterest`. A drawn ellipse is approximated as a polygon on export. The regions are static; this interface does not automatically track a moving object.

In [ ]:
##### Load the Named Regions with `movement.roi` ########################################

rois = movement.roi.load_rois(ROIS_FILE)
display(pd.DataFrame({"Name": [roi.name for roi in rois],
                      "Type": [type(roi).__name__ for roi in rois]}))

## ROI Overlay Plotting Function

In [ ]:
##### Plot Saved Regions of Interest on Ax ##############################################


def plot_ax_rois(
    ax: plt.Axes,
    rois: list,
    background_frame: np.ndarray,
    subtitle: str | None = "Drawn Regions of Interest",
    **kwargs,
) -> plt.Axes:
    """Display already-defined movement ROIs over their matching video frame.

    Parameters
    ----------
    ax : plt.Axes
        Axis on which to draw the overlay.
    rois : list
        movement LineOfInterest or PolygonOfInterest objects in video coordinates.
    background_frame : np.ndarray
        Unscaled RGB frame on which the regions were drawn.
    subtitle : str or None
        Overlay title; None omits it.
    **kwargs
        Passed to each region's plot method, e.g. linewidth and alpha.

    Returns
    -------
    plt.Axes
        Supplied axis with the video and named regions.
    """
    #==== 1| Show the Video Without Resizing Its Pixel Coordinates ========================
    height, width = background_frame.shape[:2]
    extent = (-0.5, width - 0.5, height - 0.5, -0.5)
    ax.imshow(background_frame, origin="upper", extent=extent)

    #==== 2| Draw Each Region Using movement's Own ROI Plot Method ========================
    for roi in rois:
        if isinstance(roi, movement.roi.PolygonOfInterest):
            roi.plot(ax=ax, facecolor="none", label=roi.name, **kwargs)
        else:
            roi.plot(ax=ax, label=roi.name, **kwargs)

    #==== 3| Preserve the Original Video Extent and Identify the Regions ==================
    ax.set(xlim=extent[:2], ylim=extent[2:], aspect="equal",
           xlabel="x (Pixels)", ylabel="y (Pixels)")
    if subtitle is not None:
        ax.set_title(subtitle)
    ax.legend()
    return ax

### Generate the ROI Capability Figure

In [ ]:
##### Save the Drawn-ROI Overlay ########################################################

fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
plot_ax_rois(ax=ax, rois=rois, background_frame=background_frame, linewidth=1.5)
fig.savefig("drawn_rois.png", dpi=200, bbox_inches="tight")
fig.savefig("drawn_rois.svg", bbox_inches="tight")
plt.show()

The SVG retains vector region outlines and text over an embedded raster video frame. It is not a claim that the video itself has been vectorised.

Sources: [movement GUI](https://movement.neuroinformatics.dev/latest/user_guide/gui.html), [ROI drawing example](https://movement.neuroinformatics.dev/latest/examples/boundary_angles.html), [save_rois](https://movement.neuroinformatics.dev/latest/api/movement.roi.save_rois.html).